# Construcción del FAISS

In [2]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer(
    "../modelos/embeddings/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
from pathlib import Path
import json

import faiss
import numpy

from sentence_transformers import SentenceTransformer

In [6]:
INPUT_FILE = Path(
    "../shared/output/financiera/chunks/todos.json"
)

OUTPUT_DIR = Path(
    "../shared/output/financiera/faiss"
)

INDEX_FILE = OUTPUT_DIR / "index.faiss"
METADATA_FILE = OUTPUT_DIR / "metadata.json"

In [7]:
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

chunks = json.loads(
    INPUT_FILE.read_text(
        encoding="utf-8"
    )
)

textos = [
    item["texto"]
    for item in chunks
]

print(
    f"Generando embeddings de "
    f"{len(textos)} chunks..."
)

embeddings = modelo.encode(
    textos,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings = numpy.asarray(
    embeddings,
    dtype="float32"
)

# ------------------------------------------
# Normalización L2
# ------------------------------------------

faiss.normalize_L2(
    embeddings
)

dimension = embeddings.shape[1]

print(
    f"Dimensión embedding: {dimension}"
)

# ------------------------------------------
# Índice por producto interno
# equivale a coseno tras normalizar
# ------------------------------------------

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    embeddings
)

print(
    f"Vectores indexados: {index.ntotal}"
)

# ------------------------------------------
# Guardar índice
# ------------------------------------------

faiss.write_index(
    index,
    str(INDEX_FILE)
)

# ------------------------------------------
# Guardar metadata paralela
# ------------------------------------------

METADATA_FILE.write_text(
    json.dumps(
        chunks,
        indent=4,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print()
print(
    f"FAISS:    {INDEX_FILE}"
)

print(
    f"Metadata: {METADATA_FILE}"
)

Generando embeddings de 194 chunks...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Dimensión embedding: 384
Vectores indexados: 194

FAISS:    ../shared/output/financiera/faiss/index.faiss
Metadata: ../shared/output/financiera/faiss/metadata.json
